# KaleidoNet: Full Training on Colab Pro (A100 GPU)

**Experiments:**
1. KaleidoNet (10K steps, 3 seeds) on CIFAR-100
2. Dense ViT baseline (10K steps, 3 seeds) on CIFAR-100
3. CIFAR-10 (both models, 3 seeds each)
4. Pruning baselines
5. Pillar ablation study
6. Sparsity sweep

**Estimated time:** ~1-2 hours on A100

In [17]:
# Cell 1: Check A100 GPU
import torch
assert torch.cuda.is_available(), 'No GPU! Go to Runtime > Change runtime type > A100 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')
device = torch.device('cuda')

GPU: NVIDIA A100-SXM4-80GB
Memory: 85.1 GB
PyTorch: 2.10.0+cu128


In [3]:
# Cell 2: Clone KaleidoNet and install
import os, sys
os.chdir('/content')

if not os.path.exists('/content/KaleidoNet/kaleidonet/__init__.py'):
    !git clone https://github.com/karimmagdy/kaleidonet-colab-src.git KaleidoNet
    assert os.path.exists('/content/KaleidoNet/kaleidonet/__init__.py'), 'Clone failed!'
    print('Clone OK')
else:
    # Pull latest changes
    os.chdir('/content/KaleidoNet')
    !git fetch origin && git reset --hard origin/main
    print('Updated to latest')

os.chdir('/content/KaleidoNet')
sys.path.insert(0, '/content/KaleidoNet')

!pip install -e '.[full]' -q
!pip install pyyaml -q
print(f'CWD: {os.getcwd()}')

Cloning into 'KaleidoNet'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 82 (delta 18), reused 79 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 74.78 KiB | 18.70 MiB/s, done.
Resolving deltas: 100% (18/18), done.
Clone OK
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
ERROR: Exception:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolver.resolve(
    

In [4]:
# Cell 3: Verify
from kaleidonet.model import KaleidoNet
from kaleidonet.training.trainer import TrainerConfig

model = KaleidoNet()
cfg = TrainerConfig()
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')
print(f'Device: {cfg.device}')
assert 'cuda' in cfg.device, f'Expected CUDA, got {cfg.device}'
print('All good — CUDA will be used for training')
del model  # free memory

Params: 43,492,506
Device: cuda
All good — CUDA will be used for training


## 1. Main Experiments: CIFAR-100 (3 seeds each)

In [5]:
# Cell 4: KaleidoNet on CIFAR-100
!python experiments/multi_seed_run.py \
    --model kaleidonet \
    --dataset cifar100 \
    --seeds 1 2 3 \
    --steps 10000


  KaleidoNet cifar100 seed=1 | seed=1 | steps=10000
  cmd: /usr/bin/python3 -u run.py experiments/baselines/train_cifar100.py --seed 1 --steps 10000

KaleidoNet CIFAR-100 Training
Seed: 1
100% 169M/169M [00:12<00:00, 13.1MB/s] 
Total params:  5,490,752
Active params: 5,490,752
Active fraction: 100.0%

Dense FLOPs (per sample):   682,444,800
Active FLOPs at init:       239,516,160
FLOPs budget (50% active):  119,758,080
Target: 50% of init active = 18% of dense

Seed set to 1
Starting KaleidoNet training on cuda
  Model params: 5,490,752
  FLOPs budget: 119,758,080
  Max steps: 10000

Step      0 | loss=4.8075 | task=4.7441 | soft=62% hard=100% | logits=[0.50,0.50,0.50] | λ=0.0100 | τ=1.000 | lr=1.20e-05 | 0.5 steps/s
Step     50 | loss=4.7913 | task=4.7375 | soft=62% hard=100% | logits=[0.47,0.50,0.53] | λ=0.0100 | τ=0.982 | lr=4.08e-05 | 6.9 steps/s
Step    100 | loss=4.5705 | task=4.5202 | soft=62% hard=100% | logits=[0.43,0.50,0.57] | λ=0.0100 | τ=0.964 | lr=1.14e-04 | 7.9 steps/s


In [6]:
# Cell 5: Dense ViT baseline on CIFAR-100
!python experiments/multi_seed_run.py \
    --model dense \
    --dataset cifar100 \
    --seeds 1 2 3 \
    --steps 10000


  Dense ViT cifar100 seed=1 | seed=1 | steps=10000
  cmd: /usr/bin/python3 -u experiments/baselines/dense_vit_baseline.py --seed 1 --steps 10000

Dense ViT Baseline — CIFAR-100
Seed: 1
/content/KaleidoNet/experiments/baselines/dense_vit_baseline.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
Total params: 1,821,028
Device: cuda
Step      0 | loss=4.7973 | lr=3.00e-04 | 2.1 steps/s
Step     50 | loss=4.2872 | lr=3.00e-04 | 22.7 steps/s
Step    100 | loss=4.0293 | lr=3.00e-04 | 25.2 steps/s
Step    150 | loss=4.2374 | lr=3.00e-04 | 26.2 steps/s
Step    200 | loss=4.1086 | lr=3.00e-04 | 26.7 steps/s
Step    250 | loss=3.9341 | lr=3.00e-04 | 26.9 steps/s
Step    300 | loss=4.0867 | lr=2.99e-04 | 27.1 steps/s
Step    350 | loss=4.2699 | lr=2.99e-04 | 27.1 steps/s
Step    400 | loss=3.5179 | lr=2.99e-04 | 27.3 steps/s
Step    450 | los

## 2. Additional Dataset: CIFAR-10

In [7]:
# Cell 6: KaleidoNet on CIFAR-10
!python experiments/multi_seed_run.py \
    --model kaleidonet \
    --dataset cifar10 \
    --seeds 1 2 3 \
    --steps 10000


  KaleidoNet cifar10 seed=1 | seed=1 | steps=10000
  cmd: /usr/bin/python3 -u run.py experiments/baselines/train_cifar10.py --seed 1 --steps 10000

KaleidoNet CIFAR-10 Training
Seed: 1
100% 170M/170M [00:13<00:00, 12.6MB/s] 
Total params:  5,473,382
Active params: 5,473,382
Active fraction: 100.0%

Dense FLOPs (per sample):   680,232,960
Active FLOPs at init:       237,269,760
FLOPs budget (50% active):  118,634,880

Seed set to 1
Starting KaleidoNet training on cuda
  Model params: 5,473,382
  FLOPs budget: 118,634,880
  Max steps: 10000

Step      0 | loss=2.4615 | task=2.3939 | soft=62% hard=100% | logits=[0.50,0.50,0.50] | λ=0.0100 | τ=1.000 | lr=1.20e-05 | 1.3 steps/s
Step     50 | loss=2.3479 | task=2.2888 | soft=62% hard=100% | logits=[0.46,0.50,0.54] | λ=0.0100 | τ=0.982 | lr=4.08e-05 | 8.3 steps/s
Step    100 | loss=1.9930 | task=1.9423 | soft=62% hard=100% | logits=[0.43,0.51,0.59] | λ=0.0100 | τ=0.964 | lr=1.14e-04 | 8.9 steps/s
Step    150 | loss=2.1031 | task=2.0521 | sof

In [8]:
# Cell 7: Dense ViT on CIFAR-10
!python experiments/multi_seed_run.py \
    --model dense \
    --dataset cifar10 \
    --seeds 1 2 3 \
    --steps 10000


  Dense ViT cifar10 seed=1 | seed=1 | steps=10000
  cmd: /usr/bin/python3 -u experiments/baselines/dense_vit_cifar10.py --seed 1 --steps 10000

Traceback (most recent call last):
  File "/content/KaleidoNet/experiments/baselines/dense_vit_cifar10.py", line 18, in <module>
    from experiments.baselines.dense_vit_baseline import DenseViT
ModuleNotFoundError: No module named 'experiments'
  FAILED (exit code 1) after 4s

  Dense ViT cifar10 seed=2 | seed=2 | steps=10000
  cmd: /usr/bin/python3 -u experiments/baselines/dense_vit_cifar10.py --seed 2 --steps 10000

Traceback (most recent call last):
  File "/content/KaleidoNet/experiments/baselines/dense_vit_cifar10.py", line 18, in <module>
    from experiments.baselines.dense_vit_baseline import DenseViT
ModuleNotFoundError: No module named 'experiments'
  FAILED (exit code 1) after 3s

  Dense ViT cifar10 seed=3 | seed=3 | steps=10000
  cmd: /usr/bin/python3 -u experiments/baselines/dense_vit_cifar10.py --seed 3 --steps 10000

Traceback

## 3. Pruning Baselines

In [10]:
# Cell 8: Pruning baselines (all methods on CIFAR-100)
!python experiments/baselines/pruning_baselines.py --baseline all --dataset cifar100 --steps 10000

Baseline: Magnitude Pruning | cifar100 | seed=1

Phase 1: Training full model for 10000 steps (no pruning)...
Seed set to 1
Starting KaleidoNet training on cuda
  Model params: 5,490,752
  FLOPs budget: 119,758,080
  Max steps: 10000

Step      0 | loss=4.8075 | task=4.7441 | soft=62% hard=100% | logits=[0.50,0.50,0.50] | λ=0.0100 | τ=1.000 | lr=1.20e-05 | 1.3 steps/s
Step    100 | loss=4.5705 | task=4.5202 | soft=62% hard=100% | logits=[0.43,0.50,0.57] | λ=0.0100 | τ=0.964 | lr=1.14e-04 | 8.9 steps/s
Step    200 | loss=4.4243 | task=4.3730 | soft=63% hard=100% | logits=[0.41,0.51,0.63] | λ=0.0100 | τ=0.928 | lr=2.74e-04 | 9.3 steps/s
Step    300 | loss=4.3116 | task=4.2615 | soft=63% hard=100% | logits=[0.41,0.51,0.68] | λ=0.0100 | τ=0.892 | lr=3.00e-04 | 9.5 steps/s
Step    400 | loss=4.1248 | task=4.0744 | soft=63% hard=100% | logits=[0.39,0.52,0.71] | λ=0.0100 | τ=0.856 | lr=3.00e-04 | 9.6 steps/s
Step    500 | loss=3.7699 | task=3.7191 | soft=63% hard=100% | logits=[0.39,0.52,0.71

## 4. Ablation Studies

In [11]:
# Cell 9: Pillar ablation
!python run.py experiments/ablations/pillar_ablation.py

KaleidoNet Pillar Ablation Study

Ablation: dense_baseline
  elastic=False, moe=False, early_exit=False
  Init active FLOPs: 175,219,200, budget (50%): 87,609,600
Starting KaleidoNet training on cuda
  Model params: 1,922,600
  FLOPs budget: 87,609,600
  Max steps: 2000

Step      0 | loss=4.8666 | task=4.8166 | soft=100% hard=100% | logits=[0.00,0.00,0.00] | λ=0.0100 | τ=0.000 | lr=1.21e-05 | 1.4 steps/s
Step    100 | loss=4.3197 | task=4.2697 | soft=100% hard=100% | logits=[0.00,0.00,0.00] | λ=0.0100 | τ=0.000 | lr=3.00e-04 | 17.4 steps/s
Step    200 | loss=4.0779 | task=4.0279 | soft=100% hard=100% | logits=[0.00,0.00,0.00] | λ=0.0100 | τ=0.000 | lr=2.98e-04 | 18.3 steps/s
Step    300 | loss=4.0652 | task=4.0152 | soft=100% hard=100% | logits=[0.00,0.00,0.00] | λ=0.0100 | τ=0.000 | lr=2.92e-04 | 18.7 steps/s
Step    400 | loss=3.5593 | task=3.5093 | soft=100% hard=100% | logits=[0.00,0.00,0.00] | λ=0.0100 | τ=0.000 | lr=2.82e-04 | 19.0 steps/s
Step    500 | loss=4.0073 | task=3.9573

In [12]:
# Cell 10: Sparsity sweep
!python run.py experiments/ablations/sparsity_sweep.py


Sparsity sweep: target=0.50 | cifar100 | seed=1
Seed set to 1
Starting KaleidoNet training on cuda
  Model params: 5,490,752
  FLOPs budget: 119,758,080
  Max steps: 5000

Step      0 | loss=4.8075 | task=4.7441 | soft=62% hard=100% | logits=[0.50,0.50,0.50] | λ=0.0100 | τ=1.000 | lr=1.20e-05 | 1.2 steps/s
Step    100 | loss=4.5705 | task=4.5202 | soft=62% hard=100% | logits=[0.43,0.50,0.57] | λ=0.0100 | τ=0.964 | lr=1.14e-04 | 8.5 steps/s
Step    200 | loss=4.4243 | task=4.3730 | soft=63% hard=100% | logits=[0.41,0.51,0.63] | λ=0.0100 | τ=0.928 | lr=2.74e-04 | 8.9 steps/s
Step    300 | loss=4.3112 | task=4.2611 | soft=63% hard=100% | logits=[0.41,0.51,0.68] | λ=0.0100 | τ=0.892 | lr=3.00e-04 | 9.1 steps/s
Step    400 | loss=4.1402 | task=4.0899 | soft=63% hard=100% | logits=[0.39,0.52,0.71] | λ=0.0100 | τ=0.856 | lr=2.99e-04 | 9.2 steps/s
Step    500 | loss=3.7555 | task=3.7049 | soft=63% hard=100% | logits=[0.39,0.52,0.71] | λ=0.0100 | τ=0.820 | lr=2.98e-04 | 9.3 steps/s
  [EVAL] va

## 5. Collect Results

In [13]:
# Cell 11: Analyze results
!python experiments/analyze_multi_seed.py

  Multi-Seed Results Analysis

--- KaleidoNet (3 seeds) ---
  seed=1: val_acc=39.22%  active_flops=132.4M  active_params=39.8%
  seed=2: val_acc=39.02%  active_flops=132.4M  active_params=39.8%
  seed=3: val_acc=39.51%  active_flops=132.4M  active_params=39.8%

  Val Accuracy:  39.25 ± 0.25%  (min=39.02, max=39.51, n=3)
  Active FLOPs:  132.4 ± 0.0M  (min=132.4, max=132.4)

--- Dense ViT (3 seeds) ---
  seed=1: val_acc=45.57%
  seed=2: val_acc=45.28%
  seed=3: val_acc=45.59%

  Val Accuracy:  45.48 ± 0.17%  (min=45.28, max=45.59, n=3)

  Summary Comparison
  Model                        Val Acc           FLOPs    Speedup
  ------------------------------------------------------------
  Dense ViT        45.48 ± 0.17%          236.7M      1.00x
  KaleidoNet       39.25 ± 0.25%    132.4 ± 0.0M    1.79x

  Accuracy retention: 86.3%
  FLOPs reduction:   1.79x

Summary saved to results/multi_seed_summary.json


In [14]:
# Cell 12: List all results
import glob
results = sorted(glob.glob('results/**/*.json', recursive=True))
print(f'Found {len(results)} result files:')
for r in results:
    print(f'  {r}')

Found 23 result files:
  results/dense_vit_cifar100_seed1.json
  results/dense_vit_cifar100_seed2.json
  results/dense_vit_cifar100_seed3.json
  results/kaleidonet_cifar100_seed1.json
  results/kaleidonet_cifar100_seed2.json
  results/kaleidonet_cifar100_seed3.json
  results/kaleidonet_cifar10_seed1.json
  results/kaleidonet_cifar10_seed2.json
  results/kaleidonet_cifar10_seed3.json
  results/linear_schedule_cifar100_seed1.json
  results/magnitude_pruning_cifar100_seed1.json
  results/mask_lr_sweep_1.0x_cifar100_seed1.json
  results/mask_lr_sweep_2.0x_cifar100_seed1.json
  results/mask_lr_sweep_3.0x_cifar100_seed1.json
  results/mask_lr_sweep_5.0x_cifar100_seed1.json
  results/multi_seed_summary.json
  results/random_pruning_cifar100_seed1.json
  results/sparsity_sweep_0.50_cifar100_seed1.json
  results/sparsity_sweep_0.60_cifar100_seed1.json
  results/sparsity_sweep_0.70_cifar100_seed1.json
  results/sparsity_sweep_0.80_cifar100_seed1.json
  results/sparsity_sweep_0.90_cifar100_seed1.

In [15]:
# Cell 13: Download results
import shutil
shutil.make_archive('kaleidonet_results', 'zip', '.', 'results')
from google.colab import files
files.download('kaleidonet_results.zip')
print('Results downloaded!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Results downloaded!


## 6. Inference Benchmarks

In [16]:
# Cell 14: Benchmark inference
!python run.py experiments/benchmarks/benchmark_inference.py

/content/KaleidoNet/experiments/baselines/dense_vit_baseline.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
Inference Benchmark
{
  "device": "cuda",
  "batch_size": 64,
  "warmup": 20,
  "steps": 100,
  "dense_vit": {
    "params": 1821028,
    "mean_ms": 2.2226571299688658,
    "median_ms": 2.2317709990602452,
    "min_ms": 2.0565519989759196,
    "max_ms": 2.31524699847796,
    "throughput_samples_per_s": 28794.36469847983
  },
  "kaleidonet": {
    "total_params": 5490752,
    "active_params": 2183760,
    "dormant_params": 3306992,
    "active_fraction": 0.3977160141270267,
    "active_flops": 132387840,
    "mean_ms": 35.05324697980541,
    "median_ms": 34.820419998141006,
    "min_ms": 34.52567600106704,
    "max_ms": 41.44169299979694,
    "throughput_samples_per_s": 1825.7937713122885
  },
  "surgery": {
    "original_par